<a href="https://colab.research.google.com/github/ahhill27/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** vendor_id

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines= pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

print(lines)
print('rows:', len(lines))
print('quantities:', lines['qty'].sum())


           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-08
rows: 4
quantities: 7


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
joined = lines.merge(pd.DataFrame(vendors_json), on='vendor_id', how='left', indicator=True)
print(joined)

missing= joined[joined['_merge']== 'left_only']
print('unmatched vendors:', missing['vendor_id'].unique())
print('units attached:', missing['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-08           NaN  NaN  left_only
unmatched vendors: ['V-08']
units attached: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
joined['name'] = joined['name'].fillna('Unmatched Vendor')
joined['zone'] = joined['zone'].fillna('Unknown')
joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,Unmatched Vendor,Unknown,left_only


EXPLANATION: The unmatched vendor is vendor 'V-08'. I chose to keep the unmatched vendor in the dataframe and name the umatched vendor as 'Unmatched Vendor'. Getting rid of the entire row would have led to us having incorrect revenue numbers, as well as incorrect numbers of items sold, giving us an incorrect report overall. Instead, I labeled the unmatched vendor and unknown zone so that we can still keep track of sold items, even though we do not have a description of the vendor in the vendors_json.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
zone_totals= joined.groupby('zone')['qty'].sum()
zone_totals

,qty
zone,
A,4
Unknown,3


EXPLANATION: Since I gave the unmatched vendor an 'Unknown' zone in Q3, the total does add up to 7.

If I did not give it the Unknown zone, the total would appear to be 4 items sold, even though there were 7. This would be due to the vendor V-08 not being included when I pulled information on zone totals.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [7]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]}, # no qty
    {'order': 7, 'vendor_id': 'V-08', 'lines': [{'item': 'Hot Dog', 'qty': 0}]}, #quantity of 0
]

for order in ragged_json:
  if 'lines' not in order or not order['lines']:
    order['lines']= [{'item':None, 'qty':None}]

df= pd.json_normalize(ragged_json, record_path= 'lines', meta=['order', 'vendor_id'])
print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-08


On this question, it asks me to produce a frame where a missing quantity is distinguishable from a quantity of 0, but there orignally was no row with a quantity of 0 so I added one.

Order 7 has a quantity of 0 but I added it myself to show how the frame distinguishes between missing quantities and quantities of 0.

### Q6 — Validate

In [8]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines)
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a. One row per line item makes it easier for us to visualize an item that was sold, its quantity, corresponding order number, and vendor_id seperately. If we have one line item per row, we are better able to analyze the things that are sold and their corresponding information. By doing one *order* per row, we run into the issue of trying to decipher what item corresponds to what values. It would also make it a lot more difficult to calculate totals or conduct further analyses.

b. A missing quantity means that there was never a quantity recorded during the process of data collection or that it is unavailable, while a quantity of zero means that there was a quantity observation recorded, and it just happened to be 0. If you conflate these two numbers, you may throw out very valuable data for your analysis that can be meaningful in the conclusions you draw. For example, if you call a 0 a missing value and delete it, you could get an inflated average quantity sold. If you do it the opposite way, then you would be treating unknown data as a definite value.
